# Week 76, Day 4: Sockets

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w076d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('netsim.py', 'w', encoding='utf-8') as f:
    f.write('"""Pretend TCP connections that run in the page, for practising sockets (Week 76).\n\na, b = socketpair()          two connected ends: bytes sent on one arrive at the other\na.send(data) -> int          sends SOME of data (perhaps not all), returning how many bytes\na.sendall(data)              (not provided: that\'s an exercise)\nb.recv(n) -> bytes           returns between 1 and n bytes, or b"" once the other end has\n                             closed and everything has been read\na.shutdown()                 says "I\'ve finished sending" (like sock.shutdown(socket.SHUT_WR)):\n                             the other end\'s recv then returns b"", but this end can still recv\na.close()                    closes this end completely\n\nLike real TCP, the connection is a stream of bytes with no message boundaries: one send can\narrive as several recvs, and several sends as one. With chunk=k, sends and recvs move at most\nk bytes at a time (chosen with a seeded random generator, so runs repeat exactly).\n"""\nimport random\nfrom collections import deque\n\n\nclass _Channel:\n    def __init__(self):\n        self.data = deque()\n        self.closed = False\n\n\nclass FakeSocket:\n    def __init__(self, inbox, outbox, chunk, rng, name):\n        self._inbox, self._outbox = inbox, outbox\n        self._chunk, self._rng, self.name = chunk, rng, name\n        self.closed = False\n        self.sent = 0\n\n    def _size(self, n):\n        if self._chunk is None:\n            return n\n        return max(1, min(n, self._rng.randint(1, self._chunk)))\n\n    def send(self, data):\n        if self.closed:\n            raise OSError("send on a closed socket")\n        if self._outbox.closed:\n            raise BrokenPipeError("the other end has closed")\n        data = bytes(data)\n        if not data:\n            return 0\n        n = self._size(len(data))\n        self._outbox.data.extend(data[:n])\n        self.sent += n\n        return n\n\n    def recv(self, bufsize):\n        if self.closed:\n            raise OSError("recv on a closed socket")\n        if bufsize < 1:\n            raise ValueError("bufsize must be positive")\n        if not self._inbox.data:\n            if self._inbox.closed:\n                return b""\n            raise BlockingIOError("no data yet: in a real program this would wait")\n        n = self._size(min(bufsize, len(self._inbox.data)))\n        return bytes(self._inbox.data.popleft() for _ in range(n))\n\n    def shutdown(self):\n        """Finish sending: the other end reads b"" after the data already sent."""\n        self._outbox.closed = True\n\n    def close(self):\n        if not self.closed:\n            self.closed = True\n            self._outbox.closed = True\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *exc):\n        self.close()\n\n    def __repr__(self):\n        return f"<FakeSocket {self.name}{\' closed\' if self.closed else \'\'}>"\n\n\ndef socketpair(chunk=None, seed=0):\n    """Two connected FakeSockets. chunk limits how many bytes one send or recv moves."""\n    rng = random.Random(seed)\n    a_to_b, b_to_a = _Channel(), _Channel()\n    return FakeSocket(b_to_a, a_to_b, chunk, rng, "a"), FakeSocket(a_to_b, b_to_a, chunk, rng, "b")\n')

**Predict before you run:** what will this print?



In [ ]:
from netsim import socketpair

client, server = socketpair(chunk=5, seed=1)
for message in [b"HELLO\n", b"ADD 2 3\n", b"QUIT\n"]:
    sent = 0
    while sent < len(message):
        sent += client.send(message[sent:])
client.close()

pieces = []
while True:
    data = server.recv(1024)
    if not data:
        break
    pieces.append(data)
print(pieces)
print(b"".join(pieces).split(b"\n"))

In [ ]:
import json, struct
from netsim import socketpair

def send_message(sock, obj):
    body = json.dumps(obj).encode("utf-8")
    data = struct.pack("!I", len(body)) + body
    while data:
        data = data[sock.send(data):]

def recv_exactly(sock, n):
    buf = b""
    while len(buf) < n:
        chunk = sock.recv(n - len(buf))
        if not chunk:
            raise ConnectionError("closed in the middle of a message")
        buf += chunk
    return buf

def recv_message(sock):
    (length,) = struct.unpack("!I", recv_exactly(sock, 4))
    return json.loads(recv_exactly(sock, length))

a, b = socketpair(chunk=3, seed=7)
send_message(a, {"op": "add", "args": [2, 3]})
send_message(a, {"op": "greet", "name": "Zoë"})
print(recv_message(b))
print(recv_message(b))
print(struct.pack("!I", 513), struct.unpack("!HB", b"\x01\x02\x03"))

> **Key idea.** A socket is one end of a connection: create_connection connects (with a timeout), send returns how many bytes it took (sendall sends them all), recv returns up to n bytes, b"" meaning the other end closed, and close ends it. TCP is a stream: sends and recvs don't line up, sends can be partial, and multi-byte characters can be split, so buffer bytes and decode after joining or incrementally. Frame messages with a delimiter (a buffer cut at newlines), a length prefix (read exactly the header, then exactly the body; struct packs numbers in network order with !), or by closing. Servers bind, listen and accept a new socket per client; real ones serve many at once with threads or asyncio. UDP sends whole datagrams without guarantees.

### Exercise: Sending everything

Write `sendall(sock, data)`: call `sock.send` until every byte of `data` has been accepted, and return the total. Then `recv_all(sock)`: receive until the other end closes (`recv` returns `b""`), returning all the bytes.

In [ ]:
def sendall(sock, data):
    ___

def recv_all(sock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sendall():\n    \"partial sends and reads\"\n    from netsim import socketpair\n    for chunk in (1, 3, 1000):\n        a, b = socketpair(chunk=chunk, seed=chunk)\n        data = bytes(range(256)) * 4\n        assert sendall(a, data) == len(data)\n        a.close()\n        assert recv_all(b) == data\n    a, b = socketpair()\n    assert sendall(a, b\"\") == 0\n    a.close()\n    assert recv_all(b) == b\"\"")

### Exercise: Exactly n bytes

Write `recv_exactly(sock, n)`: return exactly `n` bytes, calling `recv` as often as needed (asking for no more than is still missing, so you never read part of the **next** message); raise `ConnectionError("connection closed after K of N bytes")` if the stream ends first. Then `read_header(sock)`: read a 6-byte header packed as `!BBI` (version, message type, body length) and return those three numbers as a tuple.

In [ ]:
import struct

def recv_exactly(sock, n):
    ___

def read_header(sock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_exactly():\n    \"reading exactly as much as asked\"\n    import struct\n    from netsim import socketpair\n    a, b = socketpair(chunk=2, seed=3)\n    a.send(b\"x\")\n    rest = struct.pack(\"!BBI\", 1, 7, 70000) + b\"next\"\n    while rest:\n        rest = rest[a.send(rest):]\n    assert recv_exactly(b, 1) == b\"x\" and read_header(b) == (1, 7, 70000) and recv_exactly(b, 4) == b\"next\"\n    data = b\"ab\"\n    while data:\n        data = data[a.send(data):]\n    a.close()\n    try:\n        recv_exactly(b, 5)\n        assert False\n    except ConnectionError as e:\n        assert str(e) == \"connection closed after 2 of 5 bytes\"")

### Exercise: Length-prefixed messages

Write `send_json(sock, obj)`: encode `obj` as UTF-8 JSON and send it with a **4-byte big-endian length prefix**, all of it (loop on `send`). Then `recv_json(sock)`: read one message and decode it, returning `None` if the stream is cleanly closed **before** a new message starts, and raising `ConnectionError` if it closes part-way through one. Messages may be up to 10 MB; refuse a longer announced length with `ValueError` before reading the body.

In [ ]:
import json, struct

MAX = 10 * 1024 * 1024

def send_json(sock, obj):
    ___

def recv_json(sock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_framed():\n    \"framing a stream\"\n    import struct\n    from netsim import socketpair\n    a, b = socketpair(chunk=3, seed=11)\n    messages = [{\"op\": \"greet\", \"name\": \"Zo\u00eb\"}, [1, 2, 3], \"\u03c0 \u2248 3.14159\", {}]\n    for m in messages:\n        send_json(a, m)\n    a.close()\n    assert [recv_json(b) for _ in messages] == messages and recv_json(b) is None\n    a, b = socketpair()\n    a.send(struct.pack(\"!I\", 50) + b'{\"cut')\n    a.close()\n    try:\n        recv_json(b)\n        assert False\n    except ConnectionError:\n        pass\n    a, b = socketpair()\n    a.send(struct.pack(\"!I\", 11 * 1024 * 1024))\n    try:\n        recv_json(b)\n        assert False\n    except ValueError:\n        pass")

### Exercise: A line reader

Write a class `LineReader(sock)` for newline-delimited text: `readline()` returns the next line as a `str` **without** its `\n` (and without a `\r` before it), or `None` at the end of the stream. It keeps a buffer of bytes received but not yet returned, calls `recv(4096)` only when the buffer holds no complete line, decodes complete lines only (so characters split between reads are fine), and if the stream ends with an unfinished last line, returns it before `None`. Also `__iter__`, yielding lines until the end.

In [ ]:
class LineReader:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"buffering and delimiters\"\n    from netsim import socketpair\n    a, b = socketpair(chunk=2, seed=5)\n    data = \"HELLO\\r\\nna\u00efve caf\u00e9\\nlast line without newline\".encode(\"utf-8\")\n    while data:\n        data = data[a.send(data):]\n    a.close()\n    reader = LineReader(b)\n    assert reader.readline() == \"HELLO\" and reader.readline() == \"na\u00efve caf\u00e9\"\n    assert list(reader) == [\"last line without newline\"] and reader.readline() is None\n    a, b = socketpair(chunk=100)\n    a.send(b\"one\\ntwo\\nthree\\n\")\n    a.close()\n    r = LineReader(b)\n    assert r.readline() == \"one\" and list(r) == [\"two\", \"three\"]")

### Exercise: A key-value protocol

Write the core of a tiny text server. `respond(store, line)` handles one command line and returns the reply line: `SET key value` (the value is the rest of the line, possibly with spaces) stores it and replies `OK`; `GET key` replies `VALUE text` or `NOT FOUND`; `DEL key` replies `DELETED` or `NOT FOUND`; `KEYS` replies `KEYS` followed by the sorted keys separated by spaces (just `KEYS` if there are none); `QUIT` replies `BYE`. Commands are case-insensitive, keys aren't; anything else replies `ERROR unknown command` (or `ERROR usage: ...` for a known command with the wrong number of words, such as `ERROR usage: GET key`). Then `serve(sock, store)`: read lines from `sock`, send each reply followed by `\n` (all of it), and stop after replying to `QUIT` or at the end of the stream, closing the socket. Return the number of commands handled.

In [ ]:
USAGE = {"SET": "SET key value", "GET": "GET key", "DEL": "DEL key", "KEYS": "KEYS", "QUIT": "QUIT"}

def respond(store, line):
    ___

def serve(sock, store):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_protocol():\n    \"commands and replies\"\n    store = {}\n    assert respond(store, \"SET greeting hello there\") == \"OK\" and store == {\"greeting\": \"hello there\"}\n    assert respond(store, \"get greeting\") == \"VALUE hello there\" and respond(store, \"GET Greeting\") == \"NOT FOUND\"\n    assert respond(store, \"KEYS\") == \"KEYS greeting\" and respond(store, \"DEL greeting\") == \"DELETED\" and respond(store, \"KEYS\") == \"KEYS\"\n    assert respond(store, \"GET\") == \"ERROR usage: GET key\" and respond(store, \"FLY away\") == \"ERROR unknown command\"\n    from netsim import socketpair\n    client, server = socketpair(chunk=4, seed=2)\n    script = b\"SET a 1\\nSET b two words\\nGET b\\nDEL a\\nKEYS\\nQUIT\\nGET a\\n\"\n    while script:\n        script = script[client.send(script):]\n    assert serve(server, {}) == 6 and server.closed\n    replies = b\"\"\n    while chunk := client.recv(100):\n        replies += chunk\n    assert replies.decode().split(\"\\n\") == [\"OK\", \"OK\", \"VALUE two words\", \"DELETED\", \"KEYS b\", \"BYE\", \"\"]")

### Exercise: A binary header

A sensor sends readings as packets: a header packed as `!2sBHI` (the two bytes `b"AT"`, a version number, the sensor id, and the payload length), then the payload, then a 4-byte CRC-32 checksum (`zlib.crc32`) of header and payload together, also big-endian. Write `pack_reading(sensor_id, payload, version=1)` and `unpack_reading(data)`, which returns `(version, sensor_id, payload)` and raises `ValueError` for the wrong magic bytes, a length that doesn't match, or a bad checksum.

In [ ]:
import struct, zlib

HEADER = struct.Struct("!2sBHI")

def pack_reading(sensor_id, payload, version=1):
    ___

def unpack_reading(data):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_packets():\n    \"struct and checksums\"\n    data = pack_reading(513, b\"21.5C\")\n    assert len(data) == 9 + 5 + 4 and data[:2] == b\"AT\" and data[3:5] == b\"\\x02\\x01\"\n    assert unpack_reading(data) == (1, 513, b\"21.5C\") and unpack_reading(pack_reading(7, b\"\", version=3)) == (3, 7, b\"\")\n    for bad in [b\"XX\" + data[2:], data[:-1], data[:10] + b\"9\" + data[11:]]:\n        try:\n            unpack_reading(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Bug hunt: One recv, one message

The receiver assumes one `recv` returns one whole message, so it prints the first piece it got, `b'HEL'`. Read until the newline that ends the message, so it prints `b'HELLO WORLD'`.

In [ ]:
from netsim import socketpair

a, b = socketpair(chunk=3, seed=4)
data = b"HELLO WORLD\n"
while data:
    data = data[a.send(data):]

message = b.recv(1024)
print(message.rstrip(b"\n"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a stream, not messages\"\n    expect_output(\"b'HELLO WORLD'\")")

### Bug hunt: A character cut in half

Each piece is decoded on its own, and one piece ends half-way through the two bytes of "é", so decoding fails with `UnicodeDecodeError`. Join the bytes before decoding (or use an incremental decoder), so this prints `café crème`.

In [ ]:
from netsim import socketpair

a, b = socketpair(chunk=4, seed=0)
data = "café crème".encode("utf-8")
while data:
    data = data[a.send(data):]
a.close()

text = ""
while chunk := b.recv(4):
    text += chunk.decode("utf-8")
print(text)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"decode whole characters\"\n    expect_output(\"caf\u00e9 cr\u00e8me\")")

### Bug hunt: A send that didn't send it all

`send` accepts only some of the bytes, and the code ignores its return value, so most of the message is never sent and this prints `b'hel'`. Keep sending the rest until everything has gone, so this prints `b'hello, world'`.

In [ ]:
from netsim import socketpair

a, b = socketpair(chunk=3, seed=12)
a.send(b"hello, world")
a.close()

received = b""
while chunk := b.recv(100):
    received += chunk
print(received)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"loop until sent\"\n    expect_output(\"b'hello, world'\")")

### Puzzle: An echo server

Arrange the lines of a real TCP echo server (for your own computer) that serves one client at a time, echoing each piece of data back until the client closes. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what recv returns once the other end has closed and everything has been read
q = ___   # struct.pack("!H", 258)
r = ___   # True or False: three sends of 5 bytes are always received as three recvs of 5 bytes
s = ___   # len("é".encode("utf-8"))
t = ___   # the socket method that waits for the next client and returns a new socket for it

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == b\"\" and q == b\"\\x01\\x02\" and r is False, \"p, q and r\"\n    assert s == 2 and t == \"accept\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: remote procedure calls

Build a small RPC protocol on length-prefixed JSON (4-byte big-endian length, then UTF-8 JSON). `RPCClient(sock)` has `send_call(method, *params)`, which sends `{"id": N, "method": ..., "params": [...]}` with ids counting from 1 and returns the id, and `results()`, which reads every reply waiting on the socket (until it would block or the stream ends; `netsim` raises `BlockingIOError` when nothing has arrived) and returns a dict of id to result, raising `RemoteError` (given) for a reply with an error. `serve_rpc(sock, functions)` reads calls until the end of the stream (the client signals it with `shutdown()`, a half-close), calls `functions[method](*params)`, and replies `{"id": N, "result": value}`, or `{"id": N, "error": "message"}` for an unknown method (`"no such method: NAME"`) or an exception (`"TypeName: text"`). Calls are processed in order; the checks send several, run the server, then read the replies.

In [ ]:
import json, struct

class RemoteError(Exception):
    pass

class RPCClient:
    ___

def serve_rpc(sock, functions):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rpc():\n    \"calls, replies and errors over a stream\"\n    from netsim import socketpair\n    client_sock, server_sock = socketpair(chunk=5, seed=21)\n    client = RPCClient(client_sock)\n    ids = [client.send_call(\"add\", 2, 3), client.send_call(\"upper\", \"zo\u00eb\"), client.send_call(\"divide\", 1, 0), client.send_call(\"fly\")]\n    assert ids == [1, 2, 3, 4]\n    client_sock.shutdown()\n    handled = serve_rpc(server_sock, {\"add\": lambda a, b: a + b, \"upper\": str.upper, \"divide\": lambda a, b: a / b})\n    assert handled == 4\n    try:\n        client.results()\n        assert False, \"the error replies should raise\"\n    except RemoteError as e:\n        assert str(e) in (\"3: ZeroDivisionError: division by zero\", \"4: no such method: fly\")\n    a, b = socketpair(chunk=3, seed=8)\n    c = RPCClient(a)\n    c.send_call(\"add\", 1, 1)\n    c.send_call(\"add\", 10, 20)\n    a.shutdown()\n    serve_rpc(b, {\"add\": lambda x, y: x + y})\n    assert c.results() == {1: 2, 2: 30}")

## Warm-up

Three messages went in; they came out as whatever pieces the "network" felt like, with the boundaries in the wrong places. That's not a fault in `netsim.py`: real TCP does exactly this, because it delivers a **stream of bytes**, not messages. Only after joining the pieces and splitting on the newlines that the protocol agreed to use do the messages reappear.

## Sockets

A **socket** is your program's end of a network connection. With TCP, the life of a client socket is:

```python
import socket

with socket.create_connection(("example.com", 80), timeout=5) as sock:
    sock.sendall(b"GET / HTTP/1.1\r\nHost: example.com\r\nConnection: close\r\n\r\n")
    reply = b""
    while chunk := sock.recv(4096):
        reply += chunk
print(reply.split(b"\r\n")[0])
```

- **`create_connection((host, port), timeout)`** looks the name up and connects (the TCP handshake from Day 1).
- **`send(data)`** hands bytes to the operating system to transmit, and returns **how many it accepted**, which may be fewer than you gave it. **`sendall(data)`** loops until everything is sent.
- **`recv(n)`** returns **up to** `n` bytes: whatever has arrived, perhaps just one. It returns **`b""`** only when the other end has closed the connection and every byte has been read: that's how you know the stream has ended.
- **`close()`** (or the `with` block) ends your side; a timeout makes `recv` raise `socket.timeout` (an `OSError`) instead of waiting forever.

Sockets carry **bytes**. Text must be encoded before sending and decoded after receiving, with an agreed encoding (UTF-8).

## A stream, not messages

Because TCP is a stream, three things are always true, and code that forgets them works on your laptop and fails in production, where the network is slower and busier:

1. **One `send` can arrive as several `recv`s**, and several sends can arrive as one. Never assume a `recv` returns one whole message.
2. **`send` may send only part.** Use `sendall`, or loop on `send`'s return value.
3. **A multi-byte character can be split** between two `recv`s: `"é"` is two bytes in UTF-8, and decoding each piece separately fails. Join the bytes first, or use an incremental decoder (`codecs.getincrementaldecoder("utf-8")()`), which keeps an unfinished character until the rest arrives.

`netsim.socketpair(chunk=k)` gives two connected ends whose `send` and `recv` move at most `k` bytes at a time, chosen at random (from a seed, so runs repeat), so your code meets these conditions every time it runs, not once in a thousand. Its `recv` raises `BlockingIOError` when no data has been sent yet, where a real socket would wait.

## Framing

To send **messages** over a stream, both ends must agree how to find where each one ends. Three common ways:

- **A delimiter**: end each message with a byte that can't appear inside it, usually a newline. Text protocols (SMTP, Redis, IRC and HTTP's headers) work this way. The receiver keeps a **buffer**: append each `recv`'s bytes, and while the buffer contains a newline, cut off one message.
- **A length prefix**: send the message's length first, in a fixed number of bytes, then the message. The receiver reads exactly the header, decodes the length, then reads **exactly** that many bytes. Binary protocols (and HTTP's `Content-Length`) work this way, and the message can contain anything.
- **Closing the connection**: the message ends when the stream does (`Connection: close` in HTTP). Simple, but one message per connection.

**`struct`** packs numbers into fixed-size bytes and back: `struct.pack("!I", 513)` gives the four bytes of 513 as an unsigned 32-bit integer in **network byte order** (`!`, big-endian), and `struct.unpack("!I", data)` reverses it. Format letters: `B` (1-byte unsigned), `H` (2 bytes), `I` (4), `Q` (8), lower case for signed; `s` for raw bytes.

## Servers

A TCP **server** waits for connections:

```python
import socket

with socket.create_server(("127.0.0.1", 8000)) as server:     # socket(), bind() and listen() in one call
    while True:
        conn, address = server.accept()                         # wait for a client, get a new socket for it
        with conn:
            data = conn.recv(1024)
            conn.sendall(b"you said: " + data)
```

- **`bind((host, port))`** claims an address and port (`127.0.0.1` for this machine only; `0.0.0.0` for every network interface); **`listen()`** starts accepting; **`accept()`** waits for the next client and returns a **new socket** just for that conversation, while the listening socket keeps waiting for others.
- This loop serves **one client at a time**. Real servers handle many at once with threads, processes or an event loop (Weeks 66 and 67): `socketserver.ThreadingTCPServer`, or `asyncio.start_server` with its `reader.readline()` and `writer.write()`, which does the buffering for you.
- **UDP** sockets (`socket.SOCK_DGRAM`) skip connections: `sendto(data, address)` sends one datagram, `recvfrom` receives one whole datagram (or nothing): message boundaries are kept, but delivery and order aren't guaranteed.

Whatever the server, its core is a function from the bytes it reads to the bytes it writes, which is the part you'll write and test today.

## Practice

These exercises use `netsim.socketpair(chunk=k)`, whose sends and recvs move at most `k` bytes at a time. Your code must work whatever the chunk sizes are.

## On your own computer

Run the echo server from the puzzle in one terminal, and talk to it from another:

```bash
python -c "import socket; s = socket.create_connection(('127.0.0.1', 8000)); s.sendall(b'hello'); print(s.recv(100))"
```

Then try `nc 127.0.0.1 8000` (netcat, if you have it) and type lines by hand. Finally, rewrite the server with `asyncio.start_server`, whose `await reader.readline()` does today's line buffering for you, and connect several clients at once.

## Quiz

## Daily challenge

## Before you go

Count how many times today your code had to loop where you'd have expected one call to be enough: every one is a bug that would only show up on a slow network. Tomorrow: **your own HTTP server**, built on exactly these pieces: reading a request from a stream, routing it, and writing a response.